# Maia Lite 355M — Stage 2 CPT | CPU → A100 | SAFE v1

**Fail-closed notebook.** Original `Maia_Lite_v3` tokenizer, corpus and checkpoint are READ ONLY. No original paths are created, cleaned or overwritten. All Stage 2 artifacts go into a separate directory.

**Execution:** CPU: cells 1–5 (mount, install, audit, preflight, materialize, seal). Change runtime to A100: **start again at Cell 1**; run cells 2, 3, 6–9. On every 24-hour restart: **start at Cell 1**, then cells 2, 3, 6–9. Never re-run materialization during GPU recovery.

**WARNING:** Source repositories and licenses must be verified manually before full collection. Dataset stream stability, enough distinct source tokens, checkpoint recovery and Colab performance require a pilot. 7B is an upper token-exposure budget, not guaranteed unique tokens. Do not purchase CUs based on estimates alone.
**Revision:** CPU preflight runs automatically; materialization requires two explicit approval flags in cell 5B. A crash between publishing a shard and its state snapshot stops safely for manual reconciliation; do not delete orphan shards blindly.


In [ ]:
# CELL 1 — AUTONOMOUS MOUNT (run first after EVERY runtime change/restart)
from google.colab import drive
from pathlib import Path
import os
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive', force_remount=False)
ORIGINAL = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3')
STAGE = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_Stage2')
BASE = ORIGINAL / 'checkpoints' / 'checkpoint-216567'
TOKENIZER = ORIGINAL / 'tokenizer'
assert ORIGINAL.is_dir() and BASE.is_dir() and TOKENIZER.is_dir(), 'Original project/checkpoint/tokenizer missing. STOP; never recreate originals.'
assert not STAGE.resolve().is_relative_to(ORIGINAL.resolve()), 'Unsafe destination'
print('Original paths READ ONLY:', ORIGINAL, '\nStage 2:', STAGE)


Mounted at /content/drive
Original paths READ ONLY: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3 
Stage 2: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_Stage2


In [ ]:
# CELL 2 — PIN DEPENDENCIES (CPU or GPU; restart runtime if pip requires it)
%pip -q install "transformers==5.17.0" "datasets>=3,<5" "accelerate>=1,<2" "tokenizers>=0.22,<0.24" "numpy<3"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 123.1 MB/s eta 0:00:00


In [ ]:
# CELL 3 — COMPLETE FROZEN TOKENIZER + BASE AUDIT (no writes to original)
import os, json, math, hashlib, shutil, time, random, glob
from pathlib import Path
import numpy as np
import torch
from transformers import GPT2TokenizerFast, GPT2LMHeadModel, GPT2Config
SEQ=1024; MICRO=16; ACCUM=2; TOKENS_PER_STEP=SEQ*MICRO*ACCUM
TARGET_STEPS=math.ceil(7_000_000_000/TOKENS_PER_STEP)
TARGET_TOKENS=TARGET_STEPS*TOKENS_PER_STEP
SEED=42
assert (TOKENIZER/'tokenizer.json').is_file()
tok=GPT2TokenizerFast.from_pretrained(str(TOKENIZER),local_files_only=True)
assert len(tok)==50257
assert (tok.bos_token_id,tok.pad_token_id,tok.eos_token_id,tok.unk_token_id,tok.mask_token_id)==(0,1,2,3,4)
KNOWN={
'The theory of general relativity describes gravity.':[621,4388,303,1432,42874,9500,14767,18],
'A inteligência artificial pode auxiliar a pesquisa científica.':[37,15964,7965,1282,19442,261,7151,11490,18],
'La inteligencia artificial es un campo de la informática.':[1403,15489,7965,412,360,4316,281,344,33713,18],
'def square(x): return x * x':[9296,7501,12,92,2755,3069,2308,4333,2308],
'E = mc²':[41,1381,285,71,17471]}
for t,expected in KNOWN.items():
    actual=tok.encode(t,add_special_tokens=False)
    assert actual==expected,(t,actual,expected)
    assert tok.decode(actual,clean_up_tokenization_spaces=False)==t
cfg=GPT2Config.from_pretrained(str(BASE),local_files_only=True)
assert (cfg.vocab_size,cfg.n_positions,cfg.n_embd,cfg.n_layer,cfg.n_head)==(50257,1024,1024,24,16)
assert cfg.eos_token_id==2 and cfg.pad_token_id==1
assert (BASE/'config.json').exists() and any((BASE/x).exists() for x in ['model.safetensors','pytorch_model.bin','model.safetensors.index.json','pytorch_model.bin.index.json'])
print('FROZEN TOKENIZER / CHECKPOINT CONFIG: PASS')
print('Target steps:',TARGET_STEPS,'tokens:',TARGET_TOKENS)


FROZEN TOKENIZER / CHECKPOINT CONFIG: PASS
Target steps: 213624 tokens: 7000031232


## CPU only — build the dataset once

Proposed exposure mix: 35% code, 35% math, 15% science, 5% each EN/PT/ES. This is a **candidate**, not a claim of verified source suitability. Sources may exhaust, change or require authentication. Review licensing and dataset quality before running full build. Data preparation never touches the original project.

**Crash safety:** after each completed shard, an atomic snapshot stores consumed-document count **and token carry**. On restart the stream is re-opened and advanced to the snapshot position; previously committed shards are verified by SHA-256. Orphan shards cause a hard stop. Corpus seal is written only after all targets are reached. No deletion or overwrite of completed shards.
**Execution:** run source preflight, then the explicit approval cell (5B). Merely defining functions does not start materialization.


In [ ]:
# CELL 4 — SOURCE REGISTRY + CPU PREFLIGHT
from datasets import load_dataset
SOURCES=[
 dict(id='code',repo='OpenCoder-LLM/opc-fineweb-code-corpus',config=None,field='text',weight=.35),
 dict(id='math',repo='open-web-math/open-web-math',config=None,field='text',weight=.35),
 dict(id='science',repo='common-pile/peS2o',config=None,field='text',weight=.15),
 dict(id='general_en',repo='HuggingFaceFW/fineweb-edu',config='sample-10BT',field='text',weight=.05),
 dict(id='general_pt',repo='epfml/FineWeb2-HQ',config='por_Latn',field='text',weight=.05),
 dict(id='general_es',repo='epfml/FineWeb2-HQ',config='spa_Latn',field='text',weight=.05)]
assert abs(sum(s['weight'] for s in SOURCES)-1)<1e-12
remaining=TARGET_STEPS*MICRO*ACCUM
for s in SOURCES[:-1]:
    s['target_sequences']=int((TARGET_STEPS*MICRO*ACCUM)*s['weight']); remaining-=s['target_sequences']
SOURCES[-1]['target_sequences']=remaining
for s in SOURCES:s['target_tokens']=s['target_sequences']*SEQ
REGISTRY_DIGEST=hashlib.sha256(json.dumps(SOURCES,sort_keys=True).encode()).hexdigest()
DATA=STAGE/'dataset_v1'; RAW=DATA/'sources'; STATES=DATA/'states'; SEAL=DATA/'SEALED.json'
def stream_source(s):
    kw={'split':'train','streaming':True}
    if s['config'] is not None:kw['name']=s['config']
    return load_dataset(s['repo'],**kw)
def preflight():
    for s in SOURCES:
        r=next(iter(stream_source(s)))
        assert isinstance(r.get(s['field']),str) and r[s['field']].strip(),(s['id'],list(r))
        print('OK',s['id'],'sample chars:',len(r[s['field']]))
print('SOURCE REGISTRY READY:', len(SOURCES), 'sources')
print('Proposed targets:')
for item in SOURCES:
    print(' ', item['id'], f"{item['target_tokens']:,} tokens", item['repo'])
print('Starting streaming preflight. Network access may take several minutes...', flush=True)
preflight()
print('PREFLIGHT PASS — no corpus data has been written yet.', flush=True)


SOURCE REGISTRY READY: 6 sources
Proposed targets:
  code 2,450,010,112 tokens OpenCoder-LLM/opc-fineweb-code-corpus
  math 2,450,010,112 tokens open-web-math/open-web-math
  science 1,050,004,480 tokens common-pile/peS2o
  general_en 350,001,152 tokens HuggingFaceFW/fineweb-edu
  general_pt 350,001,152 tokens epfml/FineWeb2-HQ
  general_es 350,004,224 tokens epfml/FineWeb2-HQ
Starting streaming preflight. Network access may take several minutes...


README.md:   0%|          | 0.00/2.79k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/510 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/510 [00:00<?, ?it/s]

OK code sample chars: 4274


...

Resolving data files:   0%|          | 0/224 [00:00<?, ?it/s]

OK general_pt sample chars: 1920


Resolving data files:   0%|          | 0/1205 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/504 [00:00<?, ?it/s]

OK general_es sample chars: 9318
PREFLIGHT PASS — no corpus data has been written yet.


In [ ]:
# CELL 5 — CRASH-SAFE MATERIALIZER (CPU ONLY)
# Uses atomic state+carry snapshots and SHA-256 checks. Never overwrites committed shards.
# A crash after shard publish but before state snapshot leaves an orphan and STOPS safely.
assert not torch.cuda.is_available(), 'Dataset creation must run on CPU runtime'
SHARD_TOKENS=8_388_608  # 16 MiB uint16; divisible by SEQ
MIN_CHARS=80
assert SHARD_TOKENS%SEQ==0

def digest_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()
def atomic_json(path,obj):
    temp=path.with_name(path.name+'.pending')
    assert not temp.exists(),f'Unresolved temporary file: {temp}'
    with open(temp,'x',encoding='utf8') as f:
        json.dump(obj,f,indent=2);f.flush();os.fsync(f.fileno())
    os.replace(temp,path)
def snap_file(s):return STATES/(s['id']+'.npz')
def load_state(s):
    p=snap_file(s)
    if not p.exists():return dict(documents_seen=0,tokens_written=0,shards=[],complete=False,registry=REGISTRY_DIGEST),np.empty(0,dtype=np.uint16)
    with np.load(p,allow_pickle=False) as z:
        state=json.loads(z['state'].tobytes().decode());carry=np.asarray(z['carry'],dtype=np.uint16).copy()
    assert state['registry']==REGISTRY_DIGEST,'Source registry changed; STOP'
    return state,carry
def save_state(s,state,carry):
    p=snap_file(s);temp=p.with_name(p.name+'.pending.npz')
    assert not temp.exists(),f'Unresolved state temporary file: {temp}'
    payload=np.frombuffer(json.dumps(state,sort_keys=True).encode(),dtype=np.uint8)
    with open(temp,'xb') as f:
        np.savez(f,state=payload,carry=np.asarray(carry,dtype=np.uint16));f.flush();os.fsync(f.fileno())
    os.replace(temp,p)
def verify_state(s,state):
    folder=RAW/s['id'];assert state['tokens_written']==sum(r['tokens'] for r in state['shards'])
    assert state['tokens_written']<=s['target_tokens']
    names={r['file'] for r in state['shards']}
    for r in state['shards']:
        f=folder/r['file'];assert f.is_file() and f.stat().st_size==r['tokens']*2,(f,'size')
        assert digest_file(f)==r['sha256'],(f,'hash')
    actual={p.name for p in folder.glob('shard-*.bin')}
    assert actual==names,('Orphan/missing shards; STOP',actual-names,names-actual)
    assert not list(folder.glob('*.pending*')),'Unresolved shard temp; inspect manually'
def materialize(s):
    assert not SEAL.exists(),'Corpus sealed; immutable'
    folder=RAW/s['id'];folder.mkdir(parents=True,exist_ok=True);STATES.mkdir(parents=True,exist_ok=True)
    state,carry=load_state(s);verify_state(s,state)
    if state['complete']:
        assert state['tokens_written']==s['target_tokens'];print('SKIP',s['id']);return
    iterator=iter(stream_source(s))
    for _ in range(state['documents_seen']):
        try:next(iterator)
        except StopIteration:raise RuntimeError('Source exhausted while replaying documents; STOP')
    buf=[carry] if len(carry) else [];count=len(carry)
    while state['tokens_written']<s['target_tokens']:
        needed=min(SHARD_TOKENS,s['target_tokens']-state['tokens_written'])
        while count<needed:
            try:r=next(iterator)
            except StopIteration:raise RuntimeError(f"Source {s['id']} exhausted; target NOT reached")
            state['documents_seen']+=1
            txt=r.get(s['field'])
            if not isinstance(txt,str) or len(txt.strip())<MIN_CHARS:continue
            ids=tok.encode(txt.strip(),add_special_tokens=False)
            if not ids:continue
            ids.append(tok.eos_token_id)
            arr=np.asarray(ids,dtype=np.uint16);buf.append(arr);count+=len(arr)
        merged=np.concatenate(buf)
        shard=merged[:needed].copy();carry=merged[needed:].copy()
        assert len(shard)==needed and len(shard)%SEQ==0
        name=f"shard-{len(state['shards']):05d}.bin";final=folder/name;temp=folder/(name+'.pending')
        assert not final.exists() and not temp.exists(),'Refusing overwrite of existing shard'
        with open(temp,'xb') as f:shard.tofile(f);f.flush();os.fsync(f.fileno())
        os.replace(temp,final)
        state['shards'].append(dict(file=name,tokens=len(shard),sha256=digest_file(final)))
        state['tokens_written']+=len(shard)
        state['complete']=state['tokens_written']==s['target_tokens']
        save_state(s,state,carry)
        print(s['id'],f"{state['tokens_written']:,}/{s['target_tokens']:,}",flush=True)
        buf=[carry] if len(carry) else [];count=len(carry)

def seal_corpus():
    assert not SEAL.exists(),'Already sealed; refusing overwrite'
    inventory={}
    for s in SOURCES:
        state,_=load_state(s);verify_state(s,state)
        assert state['complete'] and state['tokens_written']==s['target_tokens'],s['id']
        inventory[s['id']]=dict(tokens=state['tokens_written'],shards=state['shards'])
    assert sum(v['tokens'] for v in inventory.values())==TARGET_TOKENS
    atomic_json(SEAL,dict(version=1,registry_digest=REGISTRY_DIGEST,sequence_length=SEQ,
                         total_tokens=TARGET_TOKENS,source_registry=SOURCES,inventory=inventory))
    print('CORPUS SEALED:',SEAL)

# Execution is in the NEXT cell. This cell only defines protected functions.
print('MATERIALIZER READY — execute the next cell to begin.', flush=True)


MATERIALIZER READY — execute the next cell to begin.


In [ ]:
# CELL 5B — EXPLICIT START / RESUME OF CPU CORPUS MATERIALIZATION
# Run after preflight and reviewing the licenses/terms of each source.
# This operation can take many hours. It NEVER deletes or overwrites original Maia files.
# If a source is inaccessible, exhausted, or has a changed schema, it STOPS.
# An orphan shard after a crash also causes a safe STOP requiring inspection.
APPROVE_SOURCE_LICENSES = True  # Set True only after verifying source licenses and terms
APPROVE_CPU_MATERIALIZATION = True  # Set True to begin or resume
assert APPROVE_SOURCE_LICENSES, 'Review source licenses and set APPROVE_SOURCE_LICENSES=True'
assert APPROVE_CPU_MATERIALIZATION, 'Set APPROVE_CPU_MATERIALIZATION=True to start'
assert not torch.cuda.is_available(), 'Use CPU runtime for materialization'
print('BEGIN CPU MATERIALIZATION', flush=True)
for idx, source in enumerate(SOURCES, 1):
    print(f'[{idx}/{len(SOURCES)}] {source["id"]} — {source["target_tokens"]:,} target tokens', flush=True)
    started = time.monotonic()
    materialize(source)
    print(f'[{idx}/{len(SOURCES)}] Finished in {(time.monotonic()-started)/3600:.2f} h', flush=True)
print('All sources completed; validating and sealing corpus...', flush=True)
seal_corpus()
print('DATASET READY FOR A100 STAGING', flush=True)


BEGIN CPU MATERIALIZATION
[1/6] code — 2,450,010,112 target tokens


Resolving data files:   0%|          | 0/510 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/510 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1538 > 1024). Running this sequence through the model will result in indexing errors


code 8,388,608/2,450,010,112
code 16,777,216/2,450,010,112
code 25,165,824/2,450,010,112
code 33,554,432/2,450,010,112
code 41,943,040/2,450,010,112
code 50,331,648/2,450,010,112
code 58,720,256/2,450,010,112
code 67,108,864/2,450,010,112
code 75,497,472/2,450,010,112
code 83,886,080/2,450,010,112
code 92,274,688/2,450,010,112
code 100,663,296/2,450,010,112
code 109,051,904/2,450,010,112
code 117,440,512/2,450,010,112
code 125,829,120/2,450,010,112
code 134,217,728/2,450,010,112
code 142,606,336/2,450,010,112
code 150,994,944/2,450,010,112
code 159,383,552/2,450,010,112
code 167,772,160/2,450,010,112
code 176,160,768/2,450,010,112
code 184,549,376/2,450,010,112
code 192,937,984/2,450,010,112
code 201,326,592/2,450,010,112
code 209,715,200/2,450,010,112
code 218,103,808/2,450,010,112
code 226,492,416/2,450,010,112
code 234,881,024/2,450,010,112
code 243,269,632/2,450,010,112
code 251,658,240/2,450,010,112
code 260,046,848/2,450,010,112
code 268,435,456/2,450,010,112
code 276,824,064/2,4

...

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/epfml/FineWeb2-HQ/resolve/c0c06e94fd3a44ae9e802b2b0fc533817601eb5e/por_Latn/000_00001.parquet
Retrying in 1s [Retry 1/5].


general_pt 92,274,688/350,001,152
general_pt 100,663,296/350,001,152
general_pt 109,051,904/350,001,152
general_pt 117,440,512/350,001,152
general_pt 125,829,120/350,001,152
general_pt 134,217,728/350,001,152
general_pt 142,606,336/350,001,152
general_pt 150,994,944/350,001,152
general_pt 159,383,552/350,001,152
general_pt 167,772,160/350,001,152
general_pt 176,160,768/350,001,152
general_pt 184,549,376/350,001,152
general_pt 192,937,984/350,001,152
general_pt 201,326,592/350,001,152
general_pt 209,715,200/350,001,152
general_pt 218,103,808/350,001,152
general_pt 226,492,416/350,001,152
general_pt 234,881,024/350,001,152
general_pt 243,269,632/350,001,152
general_pt 251,658,240/350,001,152
general_pt 260,046,848/350,001,152
general_pt 268,435,456/350,001,152
general_pt 276,824,064/350,001,152
general_pt 285,212,672/350,001,152
general_pt 293,601,280/350,001,152
general_pt 301,989,888/350,001,152
general_pt 310,378,496/350,001,152
general_pt 318,767,104/350,001,152
general_pt 327,155,71

Resolving data files:   0%|          | 0/1205 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/504 [00:00<?, ?it/s]

general_es 8,388,608/350,004,224
general_es 16,777,216/350,004,224
general_es 25,165,824/350,004,224
general_es 33,554,432/350,004,224
general_es 41,943,040/350,004,224
general_es 50,331,648/350,004,224
general_es 58,720,256/350,004,224
general_es 67,108,864/350,004,224
general_es 75,497,472/350,004,224
general_es 83,886,080/350,004,224
general_es 92,274,688/350,004,224
general_es 100,663,296/350,004,224
general_es 109,051,904/350,004,224
general_es 117,440,512/350,004,224
general_es 125,829,120/350,004,224
general_es 134,217,728/350,004,224
general_es 142,606,336/350,004,224
general_es 150,994,944/350,004,224
general_es 159,383,552/350,004,224
general_es 167,772,160/350,004,224
general_es 176,160,768/350,004,224
general_es 184,549,376/350,004,224
general_es 192,937,984/350,004,224
general_es 201,326,592/350,004,224
general_es 209,715,200/350,004,224
general_es 218,103,808/350,004,224
general_es 226,492,416/350,004,224
general_es 234,881,024/350,004,224
general_es 243,269,632/350,004,2

## A100 — autonomous staging and restart

Run Cell 1 → 2 → 3 again after switching runtimes. **Do not run CPU materialization on GPU.** Cell 6 independently mounts Drive, reads the frozen registry from the sealed manifest, verifies every shard SHA-256, and copies to ephemeral SSD. It never deletes Drive data. It fails on insufficient disk or invalid seals. Dataset sampling is deterministic by global example index, not a contiguous one-pass consumption of the 7B unique tokens.

In [ ]:
# CELL 6 — AUTONOMOUS GPU RECOVERY / STAGE TO SSD (run after EVERY 24h restart)
# Standalone: imports, mount, source registry, integrity checks, copy. No variables from earlier cells required.
from google.colab import drive
from pathlib import Path
import os,json,hashlib,shutil,math
import numpy as np
if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive',force_remount=False)
ROOT=Path('/content/drive/MyDrive/Colab_LLMs')
BASE=ROOT/'Maia_Lite_v3'/'checkpoints'/'checkpoint-216567'
TOKENIZER=ROOT/'Maia_Lite_v3'/'tokenizer'
STAGE=ROOT/'Maia_Lite_Stage2'
SEAL=STAGE/'dataset_v1'/'SEALED.json'
LOCAL=Path('/content/maia_stage2_data')
assert BASE.is_dir() and TOKENIZER.is_dir() and SEAL.is_file(),'Missing original or sealed dataset; STOP'
meta=json.loads(SEAL.read_text())
assert meta['version']==1 and meta['sequence_length']==1024
assert hashlib.sha256(json.dumps(meta['source_registry'],sort_keys=True).encode()).hexdigest()==meta['registry_digest']
SOURCES=meta['source_registry'];SEQ=1024;MICRO=16;ACCUM=2;SEED=42
TOKENS_PER_STEP=SEQ*MICRO*ACCUM
TARGET_STEPS=math.ceil(7_000_000_000/TOKENS_PER_STEP)
assert meta['total_tokens']==TARGET_STEPS*TOKENS_PER_STEP

def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()

# Inventory must match all source directories. Never trust only the filenames.
expected_bytes=0
for s in SOURCES:
    inv=meta['inventory'][s['id']]
    assert inv['tokens']==s['target_tokens']
    assert sum(x['tokens'] for x in inv['shards'])==inv['tokens']
    expected_bytes+=inv['tokens']*2
    folder=STAGE/'dataset_v1'/'sources'/s['id']
    assert {p.name for p in folder.glob('shard-*.bin')}=={r['file'] for r in inv['shards']}
    for r in inv['shards']:
        p=folder/r['file']
        assert p.is_file() and p.stat().st_size==r['tokens']*2
# Check available space for files missing locally, with margin.
missing=sum(r['tokens']*2 for s in SOURCES for r in meta['inventory'][s['id']]['shards']
            if not (LOCAL/s['id']/r['file']).is_file())
LOCAL.mkdir(parents=True,exist_ok=True)
free=shutil.disk_usage(LOCAL).free
assert free>missing+3*1024**3,(f'Insufficient local SSD: need {missing/2**30:.2f} GiB plus 3 GiB reserve, free {free/2**30:.2f} GiB')
for s in SOURCES:
    target=LOCAL/s['id'];target.mkdir(parents=True,exist_ok=True)
    for r in meta['inventory'][s['id']]['shards']:
        src=STAGE/'dataset_v1'/'sources'/s['id']/r['file'];dst=target/r['file']
        if dst.exists():
            assert dst.stat().st_size==r['tokens']*2 and file_sha(dst)==r['sha256'],('Local file invalid; STOP',dst)
        else:
            assert file_sha(src)==r['sha256'],('Drive file invalid; STOP',src)
            temp=dst.with_name(dst.name+'.copying')
            assert not temp.exists(),('Unresolved partial copy; STOP',temp)
            shutil.copyfile(src,temp)
            assert temp.stat().st_size==r['tokens']*2 and file_sha(temp)==r['sha256']
            os.replace(temp,dst)
    print('STAGED',s['id'])
print('ALL SHARDS VERIFIED ON LOCAL SSD:',LOCAL)


STAGED code
STAGED math
STAGED science
STAGED general_en
STAGED general_pt
STAGED general_es
ALL SHARDS VERIFIED ON LOCAL SSD: /content/maia_stage2_data


In [11]:
# CELL 7 — TRAINING DATASET, CONFIG, FULL-STATE RESUME CHECK
import torch, numpy as np, hashlib, time
from torch.utils.data import Dataset
from transformers import GPT2LMHeadModel, GPT2TokenizerFast, TrainingArguments, Trainer
assert torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(0).upper(),'A100 required'
assert torch.cuda.is_bf16_supported()
checktok=GPT2TokenizerFast.from_pretrained(str(TOKENIZER),local_files_only=True)
assert len(checktok)==50257 and (checktok.pad_token_id,checktok.eos_token_id)==(1,2)
assert checktok.encode('def square(x): return x * x',add_special_tokens=False)==[9296,7501,12,92,2755,3069,2308,4333,2308]

class StableDataset(Dataset):
    def __init__(self,local,meta,steps):
        self.length=steps*MICRO*ACCUM
        self.sources=meta['source_registry']
        self.cutoffs=np.cumsum([s['weight'] for s in self.sources])
        self.arrays={};self.offsets={};self.totals={}
        for s in self.sources:
            name=s['id'];arr=[];offs=[0]
            for r in meta['inventory'][name]['shards']:
                a=np.memmap(local/name/r['file'],dtype=np.uint16,mode='r')
                assert len(a)==r['tokens'] and len(a)%SEQ==0
                arr.append(a);offs.append(offs[-1]+len(a)//SEQ)
            assert offs[-1]>0
            self.arrays[name]=arr;self.offsets[name]=np.asarray(offs,dtype=np.int64);self.totals[name]=offs[-1]
    def __len__(self):return self.length
    def __getitem__(self,index):
        digest=hashlib.blake2b(f'{SEED}:{index}'.encode(),digest_size=16).digest()
        a=int.from_bytes(digest[:8],'little');b=int.from_bytes(digest[8:],'little')
        u=(a+.5)/2**64
        name=self.sources[min(int(np.searchsorted(self.cutoffs,u,side='right')),len(self.sources)-1)]['id']
        j=b%self.totals[name];offsets=self.offsets[name]
        k=int(np.searchsorted(offsets,j,side='right')-1);start=int((j-offsets[k])*SEQ)
        ids=np.asarray(self.arrays[name][k][start:start+SEQ],dtype=np.int64).copy()
        return {'input_ids':torch.from_numpy(ids),'labels':torch.from_numpy(ids.copy())}

RUN_MODE='full'  # 'pilot' or 'full'. Full is an INDEPENDENT run, not a changed pilot scheduler.
assert RUN_MODE in ('pilot','full')
MAX_STEPS=1000 if RUN_MODE=='pilot' else TARGET_STEPS
RUN_DIR=STAGE/'checkpoints'/RUN_MODE
RUN_DIR.mkdir(parents=True,exist_ok=True)

def valid_checkpoints(folder):
    good=[]
    for p in folder.glob('checkpoint-*'):
        if not p.is_dir():continue
        try:step=int(p.name.split('-')[-1])
        except ValueError:continue
        required=['trainer_state.json','optimizer.pt','scheduler.pt','rng_state.pth']
        if not all((p/x).is_file() for x in required):continue
        if not ((p/'model.safetensors').is_file() or (p/'pytorch_model.bin').is_file() or (p/'model.safetensors.index.json').is_file()):continue
        good.append((step,p))
    return sorted(good)

found=valid_checkpoints(RUN_DIR)
resume=str(found[-1][1]) if found else None
EXPECT_RESUME=False  # SET TRUE on any restart of a previously begun run
assert not EXPECT_RESUME or resume is not None,'Expected a complete checkpoint; STOP instead of restarting from base'
assert not (RUN_DIR/'RUN_STARTED').exists() or resume is not None,'Run previously started but no valid checkpoint; STOP'
model=GPT2LMHeadModel.from_pretrained(str(BASE),torch_dtype=torch.bfloat16,local_files_only=True)
assert sum(p.numel() for p in model.parameters())==354823168
model.config.use_cache=False
train_dataset=StableDataset(LOCAL,meta,MAX_STEPS)
print('MODE:',RUN_MODE,'STEPS:',MAX_STEPS,'RESUME:',resume,'DATA:',LOCAL)


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

MODE: full STEPS: 213624 RESUME: None DATA: /content/maia_stage2_data


In [12]:
# CELL 8 — TRAINER + SAFE APPROVAL (do not edit original checkpoint)
from transformers import TrainerCallback
class RuntimeMetrics(TrainerCallback):
    def on_train_begin(self,args,state,control,**kwargs):self.t0=time.monotonic();self.s0=state.global_step
    def on_log(self,args,state,control,logs=None,**kwargs):
        dt=time.monotonic()-self.t0
        print('SESSION hours:',round(dt/3600,3),'steps/s:',round((state.global_step-self.s0)/max(dt,1),3))

args=TrainingArguments(
    output_dir=str(RUN_DIR),per_device_train_batch_size=MICRO,gradient_accumulation_steps=ACCUM,
    max_steps=MAX_STEPS,learning_rate=5e-5,lr_scheduler_type='cosine',warmup_steps=100 if RUN_MODE=='pilot' else 2000,
    weight_decay=0.1,max_grad_norm=1.0,bf16=True,fp16=False,tf32=True,
    logging_strategy='steps',logging_steps=25,
    save_strategy='steps',save_steps=250 if RUN_MODE=='pilot' else 1000,
    save_total_limit=3,save_only_model=False,
    report_to='none',remove_unused_columns=False,dataloader_num_workers=2,dataloader_pin_memory=True,
    dataloader_drop_last=True,seed=SEED,data_seed=SEED,ignore_data_skip=False)
trainer=Trainer(model=model,args=args,train_dataset=train_dataset,callbacks=[RuntimeMetrics()])
print('Ready. Confirm the Colab CU meter, run mode, and verified SSD corpus.')


Ready. Confirm the Colab CU meter, run mode, and verified SSD corpus.


In [ ]:
# CELL 9 — EXPLICIT TRAIN GATE
APPROVE_TRAINING=True  # Set True only after pilot checks
assert APPROVE_TRAINING,'Training is intentionally disabled until explicit approval'
marker=RUN_DIR/'RUN_STARTED'
if not marker.exists():
    assert resume is None
    with open(marker,'x') as f:f.write('Stage 2 started; no automatic restart without a complete checkpoint.\n')
else:
    assert resume is not None,'Previous run detected but no full-state checkpoint; STOP'
t0=time.monotonic();s0=found[-1][0] if found else 0
result=trainer.train(resume_from_checkpoint=resume)
dt=time.monotonic()-t0
print(result)
print('Session hours:',dt/3600,'Steps/s:',(trainer.state.global_step-s0)/max(dt,1))
print('Tokens exposed this session:',(trainer.state.global_step-s0)*TOKENS_PER_STEP)
print('Full-state checkpoints on Drive:',RUN_DIR)
print('IMPORTANT: Verify optimizer, scheduler, RNG, Trainer state and model files before stopping.')


Step,Training Loss


## Known limitations / required preflight

- Streaming sources may change between restarts. Snapshot document offsets do **not** cryptographically pin the upstream dataset. For strict reproducibility, pin dataset revisions and/or stage raw documents first. The pipeline fails on missing/orphan files, but cannot detect a remote stream reordered upstream.
- A crash between atomic shard publication and snapshot commit leaves an orphan shard; **the next run stops deliberately**. Manual reconciliation is required; it will not delete files automatically.
- A 7B-token materialization can be slow, and a source may not have enough distinct content. Token exposure in training uses deterministic sampling **with replacement**; it is not proof of 7B unique examples.
- No independent held-out benchmark is included in this notebook. Create frozen math/code and EN/PT/ES evaluations before the full run. The pilot is for operational safety and cost measurement, not a capability claim.
- Google Drive checkpoint writes may be slow/non-atomic at directory level. Validate full-state recovery with a short stop/restart test before a 24-hour production run. Keep external milestone backups.
- Source licenses and access must be reviewed. No automatic source substitution.


In [10]:
from pathlib import Path

checkpointDir = Path(
    "/content/drive/MyDrive/Colab_LLMs/"
    "Maia_Lite_Stage2/checkpoints/pilot"
)

requiredFiles = [
    "trainer_state.json",
    "optimizer.pt",
    "scheduler.pt",
    "rng_state.pth",
]

modelFiles = [
    "model.safetensors",
    "model.safetensors.index.json",
    "pytorch_model.bin",
]

checkpoints = sorted(
    checkpointDir.glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[-1])
)

assert checkpoints, "No pilot checkpoints found"

for checkpoint in checkpoints:
    missing = [
        name for name in requiredFiles
        if not (checkpoint / name).is_file()
    ]

    hasModel = any(
        (checkpoint / name).is_file()
        for name in modelFiles
    )

    status = "OK" if not missing and hasModel else "INCOMPLETE"

    print(f"\n{checkpoint.name}: {status}")
    print("  Model:", "OK" if hasModel else "MISSING")
    print("  Missing:", missing or "None")


checkpoint-500: OK
  Model: OK
  Missing: None

checkpoint-750: OK
  Model: OK
  Missing: None

checkpoint-1000: OK
  Model: OK
  Missing: None
